In [ ]:
%pip install optuna optuna-dashboard

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1️⃣ Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
import pickle
import json
from pathlib import Path

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ─── Constants ──────────────────────────────────────────────────────────────
SEED = 42
TEST_SIZE = 0.2
N_SPLITS = 5
N_TRIALS = 100  # per model
N_JOBS = -1

LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Undefined']
NUM_CLASSES = len(LABEL_COLS)

# Embedding paths
EMB_DIR_CANDIDATES = ["./final_embeddings", "./Embeddings"]
EMB_DIR = next((d for d in EMB_DIR_CANDIDATES if os.path.isdir(d)), None)
if not EMB_DIR:
    raise FileNotFoundError(f"No embeddings directory found in {EMB_DIR_CANDIDATES}")

RDKIT_FILE = "rdkit_descriptors.csv"
MACCS_FILE = "maccs.csv"
MOL2VEC_FILE = "mol2vec.csv"

np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("✅ Imports loaded")
print(f"   Embeddings dir: {EMB_DIR}")
print(f"   N-Splits: {N_SPLITS}, N-Trials: {N_TRIALS}, Seed: {SEED}")

✅ Imports loaded
   Embeddings dir: ./final_embeddings
   N-Splits: 5, N-Trials: 100, Seed: 42


## 2️⃣ Load Data & Apply Variance Threshold

In [ ]:
# Load embeddings
rdkit_path = os.path.join(EMB_DIR, RDKIT_FILE)
maccs_path = os.path.join(EMB_DIR, MACCS_FILE)
mol2vec_path = os.path.join(EMB_DIR, MOL2VEC_FILE)

df_rdkit = pd.read_csv(rdkit_path)
df_maccs = pd.read_csv(maccs_path)
df_mol2vec = pd.read_csv(mol2vec_path)

print(f"📦 Loaded RDKit: {df_rdkit.shape}")
print(f"📦 Loaded MACCS: {df_maccs.shape}")
print(f"📦 Loaded Mol2Vec: {df_mol2vec.shape}")

# Validate alignment
if not (len(df_rdkit) == len(df_maccs) == len(df_mol2vec)):
    raise ValueError("Row count mismatch across embeddings")

# Validate label columns
for df_name, df in [("RDKit", df_rdkit), ("MACCS", df_maccs), ("Mol2Vec", df_mol2vec)]:
    missing_labels = [c for c in LABEL_COLS if c not in df.columns]
    if missing_labels:
        raise KeyError(f"{df_name} missing label columns: {missing_labels}")

# Extract labels (use RDKit as reference)
labels = df_rdkit[LABEL_COLS].values.argmax(axis=1)
y = labels

print(f"\n   Labels shape: {y.shape}")
print(f"   Class distribution:")
for cls_idx, cls_name in enumerate(LABEL_COLS):
    cnt = (y == cls_idx).sum()
    print(f"     {cls_name}: {cnt} ({100*cnt/len(y):.1f}%)")

# Extract embedding-specific features
rdkit_feat_cols = [c for c in df_rdkit.columns if c not in LABEL_COLS]
maccs_feat_cols = [c for c in df_maccs.columns if c not in LABEL_COLS]
mol2vec_feat_cols = [c for c in df_mol2vec.columns if c not in LABEL_COLS]

if len(rdkit_feat_cols) == 0:
    raise ValueError("RDKit has no feature columns after removing labels")
if len(maccs_feat_cols) == 0:
    raise ValueError("MACCS has no feature columns after removing labels")
if len(mol2vec_feat_cols) == 0:
    raise ValueError("Mol2Vec has no feature columns after removing labels")

X_rdkit_full = df_rdkit[rdkit_feat_cols].values
X_maccs_full = df_maccs[maccs_feat_cols].values
X_mol2vec_full = df_mol2vec[mol2vec_feat_cols].values

print(f"\n   RDKit: {X_rdkit_full.shape} (features={len(rdkit_feat_cols)})")
print(f"   MACCS: {X_maccs_full.shape} (features={len(maccs_feat_cols)})")
print(f"   Mol2Vec: {X_mol2vec_full.shape} (features={len(mol2vec_feat_cols)})")

# Apply Variance Threshold to RDKit and MACCS
vt = VarianceThreshold(threshold=0.01)

print(f"\n🔍 Applying Variance Threshold (threshold=0.01)...")
X_rdkit_vt = vt.fit_transform(X_rdkit_full)
print(f"   RDKit: {X_rdkit_full.shape[1]} → {X_rdkit_vt.shape[1]} features")

X_maccs_vt = vt.fit_transform(X_maccs_full)
print(f"   MACCS: {X_maccs_full.shape[1]} → {X_maccs_vt.shape[1]} features")


# Mol2Vec: use full features
X_mol2vec = X_mol2vec_full
print(f"   Mol2Vec: {X_mol2vec.shape[1]} features (no filtering)")

# Train-test split (stratified)
from sklearn.model_selection import train_test_split

idx_train, idx_test, y_train, y_test = train_test_split(
    np.arange(len(y)), y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

X_rdkit_train = X_rdkit_vt[idx_train]
X_rdkit_test = X_rdkit_vt[idx_test]

X_maccs_train = X_maccs_vt[idx_train]
X_maccs_test = X_maccs_vt[idx_test]

X_mol2vec_train = X_mol2vec[idx_train]
X_mol2vec_test = X_mol2vec[idx_test]

print(f"\n✅ Train/test split: train={len(y_train)}, test={len(y_test)}")

📦 Loaded RDKit: (14716, 218)
📦 Loaded MACCS: (14716, 166)
📦 Loaded Mol2Vec: (14716, 305)

   Labels shape: (14716,)
   Class distribution:
     Sweet: 9422 (64.0%)
     Bitter: 1515 (10.3%)
     Umami: 192 (1.3%)
     Sour: 1534 (10.4%)
     Undefined: 2053 (14.0%)


KeyError: "None of [Index(['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex',\n       'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt',\n       'NumValenceElectrons',\n       ...\n       'fr_sulfide', 'fr_sulfonamd', 'fr_sulfone', 'fr_term_acetylene',\n       'fr_tetrazole', 'fr_thiazole', 'fr_thiocyan', 'fr_thiophene',\n       'fr_unbrch_alkane', 'fr_urea'],\n      dtype='str', length=213)] are in the [columns]"

## 3️⃣ Define Metrics & Objective Functions

In [ ]:
# ─── Metrics ────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba=None):
    """Compute classification metrics."""
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    metrics = {
        "ACC": acc,
        "PRE": precision,
        "SENS": recall,
        "F1": f1,
        "MCC": mcc,
    }
    
    if y_proba is not None and y_proba.shape[1] == 2:
        try:
            auc = roc_auc_score(y_true, y_proba[:, 1])
            metrics["AUC"] = auc
        except:
            pass
    
    return metrics


# ─── XGBoost Objective ──────────────────────────────────────────────────────
def objective_xgboost(trial, X, y, embedding_name):
    """
    Optuna objective function for XGBoost hyperparameter tuning.
    Uses Stratified K-Fold cross-validation and out-of-fold evaluation.
    """
    # Hyperparameter space
    n_estimators = trial.suggest_int("n_estimators", 200, 600)
    max_depth = trial.suggest_int("max_depth", 3, 7)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.1, log=True)
    subsample = trial.suggest_float("subsample", 0.7, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.7, 1.0)
    
    # Stratified K-Fold CV
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    cv_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_fold_train, X_fold_val = X[train_idx], X[val_idx]
        y_fold_train, y_fold_val = y[train_idx], y[val_idx]
        
        # Train model
        model = XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            random_state=SEED,
            n_jobs=N_JOBS,
            eval_metric="mlogloss" if len(np.unique(y)) > 2 else "logloss",
            use_label_encoder=False,
            verbosity=0,
        )
        model.fit(X_fold_train, y_fold_train, verbose=False)
        
        # Evaluate on validation fold
        y_pred = model.predict(X_fold_val)
        fold_f1 = f1_score(y_fold_val, y_pred, average="macro", zero_division=0)
        cv_scores.append(fold_f1)
        
        # Report for pruning
        trial.report(np.mean(cv_scores), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return np.mean(cv_scores)


# ─── Extra Trees Objective ──────────────────────────────────────────────────
def objective_extra_trees(trial, X, y, embedding_name):
    """
    Optuna objective function for Extra Trees hyperparameter tuning.
    """
    n_estimators = trial.suggest_int("n_estimators", 200, 800)
    max_depth = trial.suggest_categorical("max_depth", [None, 10, 15, 20, 25, 30])
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    cv_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_fold_train, X_fold_val = X[train_idx], X[val_idx]
        y_fold_train, y_fold_val = y[train_idx], y[val_idx]
        
        model = ExtraTreesClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            max_features=max_features,
            random_state=SEED,
            n_jobs=N_JOBS,
        )
        model.fit(X_fold_train, y_fold_train)
        
        y_pred = model.predict(X_fold_val)
        fold_f1 = f1_score(y_fold_val, y_pred, average="macro", zero_division=0)
        cv_scores.append(fold_f1)
        
        trial.report(np.mean(cv_scores), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return np.mean(cv_scores)


# ─── SVM Objective ──────────────────────────────────────────────────────────
def objective_svm(trial, X, y, embedding_name):
    """
    Optuna objective function for SVM hyperparameter tuning.
    """
    C = trial.suggest_float("C", 0.1, 100, log=True)
    gamma = trial.suggest_categorical("gamma", ["scale", 0.001, 0.01])
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    cv_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_fold_train, X_fold_val = X[train_idx], X[val_idx]
        y_fold_train, y_fold_val = y[train_idx], y[val_idx]
        
        # Scale features for SVM
        scaler = StandardScaler()
        X_fold_train_scaled = scaler.fit_transform(X_fold_train)
        X_fold_val_scaled = scaler.transform(X_fold_val)
        
        model = SVC(
            C=C,
            kernel="rbf",
            gamma=gamma,
            random_state=SEED,
            probability=True,
        )
        model.fit(X_fold_train_scaled, y_fold_train)
        
        y_pred = model.predict(X_fold_val_scaled)
        fold_f1 = f1_score(y_fold_val, y_pred, average="macro", zero_division=0)
        cv_scores.append(fold_f1)
        
        trial.report(np.mean(cv_scores), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return np.mean(cv_scores)


print("✅ Objective functions defined")

## 4️⃣ Hyperparameter Tuning: XGBoost

In [ ]:
print("="*80)
print("🔍 Tuning XGBoost on RDKit Features")
print("="*80)

sampler_xgb = TPESampler(seed=SEED)
pruner_xgb = MedianPruner(n_warmup_steps=5)

study_xgb_rdkit = optuna.create_study(
    direction="maximize",
    sampler=sampler_xgb,
    pruner=pruner_xgb,
)

study_xgb_rdkit.optimize(
    lambda trial: objective_xgboost(trial, X_rdkit_train, y_train, "RDKit"),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial_xgb_rdkit = study_xgb_rdkit.best_trial
print(f"\n✅ Best XGBoost (RDKit) CV F1: {best_trial_xgb_rdkit.value:.4f}")
print(f"   Best hyperparameters: {best_trial_xgb_rdkit.params}")

# Repeat for MACCS and Mol2Vec
print("\n" + "="*80)
print("🔍 Tuning XGBoost on MACCS Features")
print("="*80)

sampler_xgb2 = TPESampler(seed=SEED)
pruner_xgb2 = MedianPruner(n_warmup_steps=5)

study_xgb_maccs = optuna.create_study(
    direction="maximize",
    sampler=sampler_xgb2,
    pruner=pruner_xgb2,
)

study_xgb_maccs.optimize(
    lambda trial: objective_xgboost(trial, X_maccs_train, y_train, "MACCS"),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial_xgb_maccs = study_xgb_maccs.best_trial
print(f"\n✅ Best XGBoost (MACCS) CV F1: {best_trial_xgb_maccs.value:.4f}")
print(f"   Best hyperparameters: {best_trial_xgb_maccs.params}")

print("\n" + "="*80)
print("🔍 Tuning XGBoost on Mol2Vec Features")
print("="*80)

sampler_xgb3 = TPESampler(seed=SEED)
pruner_xgb3 = MedianPruner(n_warmup_steps=5)

study_xgb_mol2vec = optuna.create_study(
    direction="maximize",
    sampler=sampler_xgb3,
    pruner=pruner_xgb3,
)

study_xgb_mol2vec.optimize(
    lambda trial: objective_xgboost(trial, X_mol2vec_train, y_train, "Mol2Vec"),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial_xgb_mol2vec = study_xgb_mol2vec.best_trial
print(f"\n✅ Best XGBoost (Mol2Vec) CV F1: {best_trial_xgb_mol2vec.value:.4f}")
print(f"   Best hyperparameters: {best_trial_xgb_mol2vec.params}")

## 5️⃣ Hyperparameter Tuning: Extra Trees

In [ ]:
print("="*80)
print("🔍 Tuning Extra Trees on RDKit Features")
print("="*80)

sampler_et = TPESampler(seed=SEED)
pruner_et = MedianPruner(n_warmup_steps=5)

study_et_rdkit = optuna.create_study(
    direction="maximize",
    sampler=sampler_et,
    pruner=pruner_et,
)

study_et_rdkit.optimize(
    lambda trial: objective_extra_trees(trial, X_rdkit_train, y_train, "RDKit"),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial_et_rdkit = study_et_rdkit.best_trial
print(f"\n✅ Best Extra Trees (RDKit) CV F1: {best_trial_et_rdkit.value:.4f}")
print(f"   Best hyperparameters: {best_trial_et_rdkit.params}")

print("\n" + "="*80)
print("🔍 Tuning Extra Trees on MACCS Features")
print("="*80)

sampler_et2 = TPESampler(seed=SEED)
pruner_et2 = MedianPruner(n_warmup_steps=5)

study_et_maccs = optuna.create_study(
    direction="maximize",
    sampler=sampler_et2,
    pruner=pruner_et2,
)

study_et_maccs.optimize(
    lambda trial: objective_extra_trees(trial, X_maccs_train, y_train, "MACCS"),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial_et_maccs = study_et_maccs.best_trial
print(f"\n✅ Best Extra Trees (MACCS) CV F1: {best_trial_et_maccs.value:.4f}")
print(f"   Best hyperparameters: {best_trial_et_maccs.params}")

print("\n" + "="*80)
print("🔍 Tuning Extra Trees on Mol2Vec Features")
print("="*80)

sampler_et3 = TPESampler(seed=SEED)
pruner_et3 = MedianPruner(n_warmup_steps=5)

study_et_mol2vec = optuna.create_study(
    direction="maximize",
    sampler=sampler_et3,
    pruner=pruner_et3,
)

study_et_mol2vec.optimize(
    lambda trial: objective_extra_trees(trial, X_mol2vec_train, y_train, "Mol2Vec"),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial_et_mol2vec = study_et_mol2vec.best_trial
print(f"\n✅ Best Extra Trees (Mol2Vec) CV F1: {best_trial_et_mol2vec.value:.4f}")
print(f"   Best hyperparameters: {best_trial_et_mol2vec.params}")

## 6️⃣ Hyperparameter Tuning: SVM

In [ ]:
print("="*80)
print("🔍 Tuning SVM on RDKit Features")
print("="*80)

sampler_svm = TPESampler(seed=SEED)
pruner_svm = MedianPruner(n_warmup_steps=5)

study_svm_rdkit = optuna.create_study(
    direction="maximize",
    sampler=sampler_svm,
    pruner=pruner_svm,
)

study_svm_rdkit.optimize(
    lambda trial: objective_svm(trial, X_rdkit_train, y_train, "RDKit"),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial_svm_rdkit = study_svm_rdkit.best_trial
print(f"\n✅ Best SVM (RDKit) CV F1: {best_trial_svm_rdkit.value:.4f}")
print(f"   Best hyperparameters: {best_trial_svm_rdkit.params}")

print("\n" + "="*80)
print("🔍 Tuning SVM on MACCS Features")
print("="*80)

sampler_svm2 = TPESampler(seed=SEED)
pruner_svm2 = MedianPruner(n_warmup_steps=5)

study_svm_maccs = optuna.create_study(
    direction="maximize",
    sampler=sampler_svm2,
    pruner=pruner_svm2,
)

study_svm_maccs.optimize(
    lambda trial: objective_svm(trial, X_maccs_train, y_train, "MACCS"),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial_svm_maccs = study_svm_maccs.best_trial
print(f"\n✅ Best SVM (MACCS) CV F1: {best_trial_svm_maccs.value:.4f}")
print(f"   Best hyperparameters: {best_trial_svm_maccs.params}")

print("\n" + "="*80)
print("🔍 Tuning SVM on Mol2Vec Features")
print("="*80)

sampler_svm3 = TPESampler(seed=SEED)
pruner_svm3 = MedianPruner(n_warmup_steps=5)

study_svm_mol2vec = optuna.create_study(
    direction="maximize",
    sampler=sampler_svm3,
    pruner=pruner_svm3,
)

study_svm_mol2vec.optimize(
    lambda trial: objective_svm(trial, X_mol2vec_train, y_train, "Mol2Vec"),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial_svm_mol2vec = study_svm_mol2vec.best_trial
print(f"\n✅ Best SVM (Mol2Vec) CV F1: {best_trial_svm_mol2vec.value:.4f}")
print(f"   Best hyperparameters: {best_trial_svm_mol2vec.params}")

## 7️⃣ Summarize Tuning Results

In [ ]:
# ─── Compile tuning results ────────────────────────────────────────────────
tuning_results = [
    {"Model": "XGBoost", "Embedding": "RDKit", "Best_CV_F1": best_trial_xgb_rdkit.value, "Params": best_trial_xgb_rdkit.params},
    {"Model": "XGBoost", "Embedding": "MACCS", "Best_CV_F1": best_trial_xgb_maccs.value, "Params": best_trial_xgb_maccs.params},
    {"Model": "XGBoost", "Embedding": "Mol2Vec", "Best_CV_F1": best_trial_xgb_mol2vec.value, "Params": best_trial_xgb_mol2vec.params},
    {"Model": "ExtraTrees", "Embedding": "RDKit", "Best_CV_F1": best_trial_et_rdkit.value, "Params": best_trial_et_rdkit.params},
    {"Model": "ExtraTrees", "Embedding": "MACCS", "Best_CV_F1": best_trial_et_maccs.value, "Params": best_trial_et_maccs.params},
    {"Model": "ExtraTrees", "Embedding": "Mol2Vec", "Best_CV_F1": best_trial_et_mol2vec.value, "Params": best_trial_et_mol2vec.params},
    {"Model": "SVM", "Embedding": "RDKit", "Best_CV_F1": best_trial_svm_rdkit.value, "Params": best_trial_svm_rdkit.params},
    {"Model": "SVM", "Embedding": "MACCS", "Best_CV_F1": best_trial_svm_maccs.value, "Params": best_trial_svm_maccs.params},
    {"Model": "SVM", "Embedding": "Mol2Vec", "Best_CV_F1": best_trial_svm_mol2vec.value, "Params": best_trial_svm_mol2vec.params},
]

results_df = pd.DataFrame(tuning_results)

print("\n" + "="*100)
print("📊 Hyperparameter Tuning Summary")
print("="*100)
print(results_df[["Model", "Embedding", "Best_CV_F1"]].to_string(index=False))

# Save tuning results
results_df.to_csv("optuna_tuning_results.csv", index=False)
print(f"\n💾 Saved: optuna_tuning_results.csv")

# Save best hyperparameters as JSON
best_params = {
    "XGBoost_RDKit": best_trial_xgb_rdkit.params,
    "XGBoost_MACCS": best_trial_xgb_maccs.params,
    "XGBoost_Mol2Vec": best_trial_xgb_mol2vec.params,
    "ExtraTrees_RDKit": best_trial_et_rdkit.params,
    "ExtraTrees_MACCS": best_trial_et_maccs.params,
    "ExtraTrees_Mol2Vec": best_trial_et_mol2vec.params,
    "SVM_RDKit": best_trial_svm_rdkit.params,
    "SVM_MACCS": best_trial_svm_maccs.params,
    "SVM_Mol2Vec": best_trial_svm_mol2vec.params,
}

with open("best_hyperparameters.json", "w") as f:
    json.dump(best_params, f, indent=2)

print("💾 Saved: best_hyperparameters.json")

## 8️⃣ Retrain Models on Full Training Data

In [ ]:
print("\n" + "="*80)
print("🔄 Retraining Models with Best Hyperparameters")
print("="*80)

trained_models = {}
scalers = {}

# ─── XGBoost Models ─────────────────────────────────────────────────────────
print("\n📦 XGBoost Models:")

print("  Training XGBoost (RDKit)...", end=" ", flush=True)
xgb_rdkit_model = XGBClassifier(**best_trial_xgb_rdkit.params, random_state=SEED, n_jobs=N_JOBS, verbosity=0, eval_metric="mlogloss")
xgb_rdkit_model.fit(X_rdkit_train, y_train)
trained_models["XGBoost_RDKit"] = xgb_rdkit_model
print("✓")

print("  Training XGBoost (MACCS)...", end=" ", flush=True)
xgb_maccs_model = XGBClassifier(**best_trial_xgb_maccs.params, random_state=SEED, n_jobs=N_JOBS, verbosity=0, eval_metric="mlogloss")
xgb_maccs_model.fit(X_maccs_train, y_train)
trained_models["XGBoost_MACCS"] = xgb_maccs_model
print("✓")

print("  Training XGBoost (Mol2Vec)...", end=" ", flush=True)
xgb_mol2vec_model = XGBClassifier(**best_trial_xgb_mol2vec.params, random_state=SEED, n_jobs=N_JOBS, verbosity=0, eval_metric="mlogloss")
xgb_mol2vec_model.fit(X_mol2vec_train, y_train)
trained_models["XGBoost_Mol2Vec"] = xgb_mol2vec_model
print("✓")

# ─── Extra Trees Models ─────────────────────────────────────────────────────
print("\n📦 Extra Trees Models:")

print("  Training Extra Trees (RDKit)...", end=" ", flush=True)
et_rdkit_model = ExtraTreesClassifier(**best_trial_et_rdkit.params, random_state=SEED, n_jobs=N_JOBS)
et_rdkit_model.fit(X_rdkit_train, y_train)
trained_models["ExtraTrees_RDKit"] = et_rdkit_model
print("✓")

print("  Training Extra Trees (MACCS)...", end=" ", flush=True)
et_maccs_model = ExtraTreesClassifier(**best_trial_et_maccs.params, random_state=SEED, n_jobs=N_JOBS)
et_maccs_model.fit(X_maccs_train, y_train)
trained_models["ExtraTrees_MACCS"] = et_maccs_model
print("✓")

print("  Training Extra Trees (Mol2Vec)...", end=" ", flush=True)
et_mol2vec_model = ExtraTreesClassifier(**best_trial_et_mol2vec.params, random_state=SEED, n_jobs=N_JOBS)
et_mol2vec_model.fit(X_mol2vec_train, y_train)
trained_models["ExtraTrees_Mol2Vec"] = et_mol2vec_model
print("✓")

# ─── SVM Models (with scaling) ──────────────────────────────────────────────
print("\n📦 SVM Models:")

print("  Training SVM (RDKit)...", end=" ", flush=True)
scaler_svm_rdkit = StandardScaler()
X_rdkit_train_scaled = scaler_svm_rdkit.fit_transform(X_rdkit_train)
svm_rdkit_model = SVC(**best_trial_svm_rdkit.params, kernel="rbf", random_state=SEED, probability=True)
svm_rdkit_model.fit(X_rdkit_train_scaled, y_train)
trained_models["SVM_RDKit"] = svm_rdkit_model
scalers["SVM_RDKit"] = scaler_svm_rdkit
print("✓")

print("  Training SVM (MACCS)...", end=" ", flush=True)
scaler_svm_maccs = StandardScaler()
X_maccs_train_scaled = scaler_svm_maccs.fit_transform(X_maccs_train)
svm_maccs_model = SVC(**best_trial_svm_maccs.params, kernel="rbf", random_state=SEED, probability=True)
svm_maccs_model.fit(X_maccs_train_scaled, y_train)
trained_models["SVM_MACCS"] = svm_maccs_model
scalers["SVM_MACCS"] = scaler_svm_maccs
print("✓")

print("  Training SVM (Mol2Vec)...", end=" ", flush=True)
scaler_svm_mol2vec = StandardScaler()
X_mol2vec_train_scaled = scaler_svm_mol2vec.fit_transform(X_mol2vec_train)
svm_mol2vec_model = SVC(**best_trial_svm_mol2vec.params, kernel="rbf", random_state=SEED, probability=True)
svm_mol2vec_model.fit(X_mol2vec_train_scaled, y_train)
trained_models["SVM_Mol2Vec"] = svm_mol2vec_model
scalers["SVM_Mol2Vec"] = scaler_svm_mol2vec
print("✓")

print("\n✅ All models trained!")

## 9️⃣ Evaluate on Test Set & Generate Probabilities

In [ ]:
print("\n" + "="*100)
print("📊 Test Set Evaluation")
print("="*100)

test_results = []
test_probabilities = {}

# ─── XGBoost Evaluation ─────────────────────────────────────────────────────
print("\n🔹 XGBoost Models:")

for model_name, model in [("XGBoost_RDKit", xgb_rdkit_model), 
                          ("XGBoost_MACCS", xgb_maccs_model), 
                          ("XGBoost_Mol2Vec", xgb_mol2vec_model)]:
    
    X_test = X_rdkit_test if "RDKit" in model_name else (X_maccs_test if "MACCS" in model_name else X_mol2vec_test)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    metrics = compute_metrics(y_test, y_pred, y_proba)
    test_results.append({**{"Model": model_name}, **metrics})
    test_probabilities[model_name] = y_proba
    
    print(f"   {model_name}: ACC={metrics['ACC']:.4f}, F1={metrics['F1']:.4f}, MCC={metrics['MCC']:.4f}")

# ─── Extra Trees Evaluation ─────────────────────────────────────────────────
print("\n🔹 Extra Trees Models:")

for model_name, model in [("ExtraTrees_RDKit", et_rdkit_model), 
                          ("ExtraTrees_MACCS", et_maccs_model), 
                          ("ExtraTrees_Mol2Vec", et_mol2vec_model)]:
    
    X_test = X_rdkit_test if "RDKit" in model_name else (X_maccs_test if "MACCS" in model_name else X_mol2vec_test)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    metrics = compute_metrics(y_test, y_pred, y_proba)
    test_results.append({**{"Model": model_name}, **metrics})
    test_probabilities[model_name] = y_proba
    
    print(f"   {model_name}: ACC={metrics['ACC']:.4f}, F1={metrics['F1']:.4f}, MCC={metrics['MCC']:.4f}")

# ─── SVM Evaluation ─────────────────────────────────────────────────────────
print("\n🔹 SVM Models:")

model_scaler_pairs = [("SVM_RDKit", svm_rdkit_model, scaler_svm_rdkit, X_rdkit_test),
                       ("SVM_MACCS", svm_maccs_model, scaler_svm_maccs, X_maccs_test),
                       ("SVM_Mol2Vec", svm_mol2vec_model, scaler_svm_mol2vec, X_mol2vec_test)]

for model_name, model, scaler, X_test in model_scaler_pairs:
    X_test_scaled = scaler.transform(X_test)
    
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    metrics = compute_metrics(y_test, y_pred, y_proba)
    test_results.append({**{"Model": model_name}, **metrics})
    test_probabilities[model_name] = y_proba
    
    print(f"   {model_name}: ACC={metrics['ACC']:.4f}, F1={metrics['F1']:.4f}, MCC={metrics['MCC']:.4f}")

# ─── Save test results ──────────────────────────────────────────────────────
test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv("optuna_test_results.csv", index=False)
print(f"\n💾 Saved: optuna_test_results.csv")

print("\n" + "="*100)
print("📊 Test Results Summary")
print("="*100)
print(test_results_df[["Model", "ACC", "F1", "PRE", "SENS", "MCC"]].to_string(index=False))

## 🔟 Save Models & Probabilities for Ensembling

In [ ]:
# Create output directory
output_dir = Path("optuna_tuned_models")
output_dir.mkdir(exist_ok=True)

# ─── Save trained models ────────────────────────────────────────────────────
print("💾 Saving trained models...")

for model_name, model in trained_models.items():
    model_path = output_dir / f"{model_name}.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"   {model_name} → {model_path}")

# ─── Save scalers for SVM ──────────────────────────────────────────────────
print("\n💾 Saving SVM scalers...")

for scaler_name, scaler in scalers.items():
    scaler_path = output_dir / f"{scaler_name}_scaler.pkl"
    with open(scaler_path, "wb") as f:
        pickle.dump(scaler, f)
    print(f"   {scaler_name}_scaler → {scaler_path}")

# ─── Save test probabilities ────────────────────────────────────────────────
print("\n💾 Saving test probabilities...")

for model_name, proba in test_probabilities.items():
    proba_path = output_dir / f"{model_name}_test_proba.npy"
    np.save(proba_path, proba)
    print(f"   {model_name}_test_proba → {proba_path}")

# Save test indices for reference
test_indices_path = output_dir / "test_indices.npy"
np.save(test_indices_path, idx_test)
print(f"   test_indices → {test_indices_path}")

# Save test labels
test_labels_path = output_dir / "test_labels.npy"
np.save(test_labels_path, y_test)
print(f"   test_labels → {test_labels_path}")

print("\n✅ All artifacts saved!")

## 1️⃣1️⃣ Final Summary Report

In [ ]:
print("\n" + "="*100)
print("🎯 OPTUNA HYPERPARAMETER TUNING SUMMARY")
print("="*100)

print("\n📋 Tuning Configuration:")
print(f"   Total Trials per Model: {N_TRIALS}")
print(f"   Cross-Validation Folds: {N_SPLITS}")
print(f"   Sampler: TPE (Tree-structured Parzen Estimator)")
print(f"   Pruner: Median Pruner")
print(f"   Embeddings Used: RDKit (VT), MACCS (VT), Mol2Vec (Full)")

print("\n📊 Best Hyperparameters by Model & Embedding:")
for _, row in results_df.iterrows():
    print(f"\n   {row['Model']} ({row['Embedding']}):")
    print(f"     CV F1: {row['Best_CV_F1']:.4f}")
    for param, value in row['Params'].items():
        print(f"     {param}: {value}")

print("\n📈 Test Set Performance:")
print(test_results_df.to_string(index=False))

print("\n🏆 Best Test F1 Scores by Model:")
for model_type in ["XGBoost", "ExtraTrees", "SVM"]:
    model_df = test_results_df[test_results_df["Model"].str.contains(model_type)]
    best_row = model_df.loc[model_df["F1"].idxmax()]
    print(f"   {model_type}: {best_row['Model']} (F1={best_row['F1']:.4f})")

print("\n💾 Output Files Generated:")
print(f"   - optuna_tuning_results.csv (tuning summary)")
print(f"   - best_hyperparameters.json (best params for each model)")
print(f"   - optuna_test_results.csv (test set metrics)")
print(f"   - optuna_tuned_models/ (trained models, scalers, probabilities)")

print("\n✅ Pipeline Complete!")